# Stock Direction Prediction — Model Overview

개장 전 공개 정보로 당일 종가 등락을 5개 등급으로 예측한 프로젝트의 최종 모델 구조와 평가 결과를 요약합니다. 실제 예측 로직은 `src/model.py`에 있습니다.

## Core idea

전일 종가 이후 이미 관측된 **장전 가격 변화**와, 과거 각 날짜에서 장전 이후 종가까지 남아 있던 **잔차 수익률의 경험분포**를 결합합니다. 예측 시점 이후 정보가 들어가지 않도록 `known_at < cutoff` 조건을 적용했습니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'evaluation').exists() else Path.cwd().parent
metrics = json.loads((ROOT / 'evaluation/metrics.json').read_text(encoding='utf-8'))
pd.DataFrame({k: metrics[k] for k in ['all', 'known', 'unseen']}).T

## Evaluation note

평가 구간은 2026-06-01~2026-09-14, 73거래일 × 50종목 = 3,650건입니다. 이 구간은 개발 과정에서 이미 확인한 적이 있어 **완전히 미사용한 독립 최종 테스트셋이 아닙니다.**

In [ ]:
classes = ['급락', '하락', '보합', '상승', '급등']
pd.DataFrame(metrics['confusion_matrix'], index=pd.Index(classes, name='실제'), columns=pd.Index(classes, name='예측'))

## Validation

최종 코드 정리 전후에 입력 피처와 예측 결과가 동일한지 확인하고, 미래 시점 데이터를 주입해도 결과가 바뀌지 않는지 점검했습니다.

In [ ]:
checks = json.loads((ROOT / 'docs/validation_results.json').read_text(encoding='utf-8'))
pd.Series({
    'feature_rows_compared': checks['feature_rows_compared'],
    'feature_dates': checks['feature_dates'],
    'feature_parity': checks['seven_features_and_guard_match_original_and_independent_rebuild'],
    'holdout_prediction_parity_rows': checks['holdout_prediction_parity_rows'],
    'submission_prediction_parity_rows': checks['submission_prediction_parity_rows'],
    'retrospective_is_independent_test': checks['retrospective_is_independent_test'],
})